# 57. RealMLP, batch size against step count. A fold-0 probe

**This is a probe and it produces no ledger row.** Fold 0 only, same treatment as `04`, `05`,
`07` and the CatBoost probe of 2026-08-04. Nothing here is comparable to a five-fold row and no
out-of-fold vector is written.

## The question

`realmlp10` sits at **0.967892** out-of-fold. The public RealMLP is at about **0.9688**, and the
audit of 2026-08-23 established that our architecture and preprocessing are already faithful:
periodic embeddings, median/IQR/smooth-clip, flat-cosine, EMA, label smoothing, parameter-group
multipliers, ten internal members. Row 138 raised the internal ensemble from 3 to 10 for +0.000164
and settled that the ensemble count is not the gap.

Two known differences remain: **the seed loop**, 1 here against their 3, and **batch size**, 512
here against theirs, which row 53's header inferred to be smaller from their runtime. This repo
has measured seed averaging as near-null three separate times, at rows 20 to 23 and row 32, so the
seed loop is the less likely of the two.

## Why a 2 by 2 and not a batch sweep

Halving the batch doubles the number of optimiser steps at fixed epochs. A plain batch sweep
therefore changes two things at once and cannot say which one paid, which is the error rows 78 to
84 caught in the CatBoost budget and the error `43` version 1 made with its projection.

| | epochs 12 | epochs 24 |
|---|---|---|
| **batch 512** | row 138's configuration | same batch, twice the steps |
| **batch 256** | half the batch, twice the steps | half the batch, four times the steps |

If step count is what matters, the two diagonal cells at 2x steps land together. If batch size
matters for its own sake, through gradient noise rather than step count, they separate.

## The prediction

**Arm A reproduces row 138's fold 0 at 0.9670 to 0.9679**, which is the check that the probe is
measuring what it claims. Beyond that, **+0.0003 to +0.0008 for the best arm**, most likely
`b256/e24`, on the reasoning that 691,369 rows at batch 512 for 12 epochs is only about 13,000
steps and that is short for this architecture.

## The case against, written first

Row 53's own case-against named the batch size as a candidate and it has beaten the prediction
eight times in ten. The specific worry here: the flat-cosine schedule and the EMA decay of
0.997875 were both tuned by the public author *for their step count*. Changing the step count
without retuning them moves the model off the configuration those constants were chosen for, and
EMA in particular has a horizon measured in steps rather than epochs. **A longer run with an EMA
horizon that no longer matches it can easily be worse, not better**, and that would show up as
`e24` losing to `e12` at both batch sizes.

The second worry is that the 0.0009 gap is simply the 150-fit tuning behind the public
configuration and is not recoverable from any single knob.

**A fold-0 probe ranks blends usefully and ranks near-identical models not at all**, per rows 9 to
12. These four arms are not near-identical, so the probe should separate them, but a difference
under about 0.0003 here should not be trusted.


In [1]:
SMOKE = True

SEED = 42
N_SPLITS = 5
N_INNER = 5
SMOOTH = 10.0
TARGET, ID = "addicted_label", "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

# Fold-0 probe. Four arms, a 2x2 over batch size and epochs. Arm A is row 138.
PROBE_FOLD = 0
PROBE_ENS = 3
ARM_GRID = [("A_b512_e12", 512, 12), ("B_b256_e12", 256, 12),
            ("C_b512_e24", 512, 24), ("D_b256_e24", 256, 24)]
ARMS = [a for a, _, _ in ARM_GRID]

# Architecture. Public configuration where it is affordable, reduced where it is not.
HIDDEN = (768, 512, 512)
DROPOUT = 0.07
EMB_DIM = 8                 # categorical embedding width
PB_HIDDEN = 32              # periodic frequencies per numeric feature
PB_OUT = 6                  # output width per numeric feature after the periodic map
PB_FREQ_SCALE = 10.0
N_ENS = 10                  # THE ONE VARIABLE. Row 135 ran 3. Public runs 10 x 3 seeds.

# Optimisation.
EPOCHS, BATCH, LR, WD = 12, 512, 8e-3, 0.015
FLAT_RATIO = 0.3            # flat-cosine: flat for this fraction, then cosine to zero
EMA_DECAY = 0.997875
LS_EPS = 0.04               # label smoothing, cosine-annealed to zero
GRAD_CLIP = 1.2
SCALE_LR_MULT, BIAS_LR_MULT = 10.0, 0.1
SCALE_WD_MULT, BIAS_WD_MULT = 0.1, 0.5

ROW127_CV = 0.965798
ROW135_CV = 0.967728
ROW138_CV = 0.967892
ROW138_FOLD0 = 0.967220
EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_ENCODER_FP = "0642e41750ef8bab"

if SMOKE:
    EPOCHS, N_SPLITS, N_ENS = 2, 2, 2
    PROBE_ENS = 2
    ARM_GRID = [(a, b, 1 if e == 12 else 2) for a, b, e in ARM_GRID]

print(f"SMOKE = {SMOKE}   n_ens {PROBE_ENS}  hidden {HIDDEN}")
print(f"probe fold {PROBE_FOLD}, arms: {[a for a, _, _ in ARM_GRID]}")
print("no validation is consulted at any point; the final EMA weights are used")

SMOKE = True   n_ens 2  hidden (768, 512, 512)
probe fold 0, arms: ['A_b512_e12', 'B_b256_e12', 'C_b512_e24', 'D_b256_e24']
no validation is consulted at any point; the final EMA weights are used


In [2]:
import ast
import hashlib
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import QuantileTransformer

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)


seed_all(SEED)
print("torch", torch.__version__, "| device", DEV)
if DEV.type != "cuda" and not SMOKE:
    print("\nWARNING: no GPU. Set the accelerator, or this takes about an hour.")

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
COLS = [c for c in train_full.columns if c not in (ID, TARGET)]
NUM_COLS = [c for c in COLS if c not in CAT_COLS]

checks = {
    "id is not a feature": ID not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full[ID]) & set(test[ID])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != ID],
}
for k, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {k}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy().astype(np.float32)

# The alignment gate. Computed locally on 2026-08-04 with scikit-learn 1.9.0. If any
# disagree, the out-of-fold vector this notebook produces cannot be blended with the
# local ones and the run is worthless.
EXPECT = {"rows": 691369, "rate": 0.709424, "sha": "ec282b0968059676",
          "sizes": [138274, 138274, 138274, 138274, 138273],
          "first20": [3, 3, 3, 4, 2, 3, 4, 0, 3, 4, 1, 1, 2, 1, 3, 1, 1, 4, 0, 3]}

folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

got = {"rows": len(train), "rate": round(float(y.mean()), 6),
       "sha": hashlib.sha256(folds.tobytes()).hexdigest()[:16],
       "sizes": np.bincount(folds).tolist(), "first20": folds[:20].tolist()}

print()
if SMOKE:
    print("  SMOKE subsamples the data, so the fold split cannot match.")
    print("  Alignment is NOT checked and this OOF is unusable.")
ALIGNED = not SMOKE
for k in ([] if SMOKE else EXPECT):
    ok = got[k] == EXPECT[k]
    ALIGNED &= ok
    print(f"  [{'ok' if ok else 'MISMATCH'}] {k}")
    if not ok:
        print(f"        expected {EXPECT[k]}")
        print(f"        got      {got[k]}")
print(f"\nfold alignment: {'verified' if ALIGNED else 'FAILED, do not use this OOF'}")
print(f"{len(train):,} train rows, {len(test):,} test rows")

torch 2.13.0+cpu | device cpu
running locally, writing to smartphone-addiction/artifacts/oof


  [ok] id is not a feature
  [ok] target is not a feature
  [ok] train and test ids do not overlap
  [ok] train and test feature lists match

  SMOKE subsamples the data, so the fold split cannot match.
  Alignment is NOT checked and this OOF is unusable.

fold alignment: FAILED, do not use this OOF
20,000 train rows, 5,000 test rows


## The encoder, fingerprinted against 13

In [3]:
X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT_COLS:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")


def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    return (hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]
            if len(parts) == len(ENCODER_FNS) else None)


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))
theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here  : {mine}")
print(f"encoder fingerprint in 13 : {theirs}")
print("encoder: IDENTICAL to rows 17, 26 and 31's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - not comparable to those rows")
ENC_COLS = [f"{p}_{c}" for c in COLS for p in ("te", "fq")]
print(f"\nnumeric block: {len(NUM_COLS)} raw + {len(ENC_COLS)} encoded = "
      f"{len(NUM_COLS) + len(ENC_COLS)} columns, plus {len(NUM_COLS)} mask columns")
print(f"row 16 had {len(NUM_COLS)} numeric + {len(NUM_COLS)} mask")

encoder fingerprint here  : 0642e41750ef8bab
encoder fingerprint in 13 : 0642e41750ef8bab
encoder: IDENTICAL to rows 17, 26 and 31's

numeric block: 9 raw + 24 encoded = 33 columns, plus 9 mask columns
row 16 had 9 numeric + 9 mask


In [4]:
# The same three leak checks 13 ran, on the same encoder. Read together: the first two
# must be about zero, the third must be large. Without the third, an encoder that
# ignored the target entirely would pass the first two and look clean.
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

import gc

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

1. flip all validation targets -> change in their encoding: 0.000e+00
2. flip 200 training rows -> change in their own encoding:  4.250e-03
   prior moved 4.250e-03, and these should track each other
3. flip all training targets -> change in val encoding:     9.270e-01

LEAK CHECKS: PASS


40

## The inputs, identical to row 127

In [5]:
DST, SM_, GM, WS = ("daily_screen_time_hours", "social_media_hours",
                    "gaming_hours", "work_study_hours")
SLP, WKD = "sleep_hours", "weekend_screen_time"
NOTIF, OPENS = "notifications_per_day", "app_opens_per_day"

RATIO_COLS = ["component_total", "slack", "weekend_lift", "weekend_ratio",
              "social_share", "gaming_share", "work_share", "sleep_minus_screen",
              "screen_to_sleep", "opens_per_hour", "notif_per_hour",
              "notif_per_open", "engagement"]


def safe_div(a, b):
    b = b.replace(0, np.nan)
    return a / b


def ratio_block(df):
    o = pd.DataFrame(index=df.index)
    o["component_total"] = df[[SM_, GM, WS]].sum(axis=1, min_count=3)
    o["slack"] = df[DST] - o["component_total"]
    o["weekend_lift"] = df[WKD] - df[DST]
    o["weekend_ratio"] = safe_div(df[WKD], df[DST])
    o["social_share"] = safe_div(df[SM_], df[DST])
    o["gaming_share"] = safe_div(df[GM], df[DST])
    o["work_share"] = safe_div(df[WS], df[DST])
    o["sleep_minus_screen"] = df[SLP] - df[DST]
    o["screen_to_sleep"] = safe_div(df[DST], df[SLP])
    o["opens_per_hour"] = safe_div(df[OPENS], df[DST])
    o["notif_per_hour"] = safe_div(df[NOTIF], df[DST])
    o["notif_per_open"] = safe_div(df[NOTIF], df[OPENS])
    o["engagement"] = df[NOTIF] + df[OPENS]
    return o.replace([np.inf, -np.inf], np.nan).astype(np.float64)


rng = np.random.default_rng(0)
_perm = train.copy()
_perm[TARGET] = rng.permutation(train[TARGET].to_numpy())
BLOCK_OK = bool(ratio_block(_perm).equals(ratio_block(train))
                and list(ratio_block(train).columns) == RATIO_COLS)
print(f"ratio block is a pure function of the features, not of y: {BLOCK_OK}")
del _perm

RB_TR = ratio_block(train).to_numpy(np.float32)
RB_TE = ratio_block(test).to_numpy(np.float32)
print(f"row 106 fed {len(NUM_COLS) + len(ENC_COLS)} numeric + {len(NUM_COLS)} mask columns")
print(f"these arms feed {len(NUM_COLS) + len(ENC_COLS) + len(RATIO_COLS)} numeric "
      f"+ {len(NUM_COLS)} mask")

cat_sizes = []
codes_tr, codes_te = {}, {}
for c in CAT_COLS:
    both = pd.concat([train[c], test[c]], ignore_index=True).astype("object")
    levels = sorted(both.dropna().unique().tolist())
    lut = {v: i + 1 for i, v in enumerate(levels)}
    codes_tr[c] = train[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    codes_te[c] = test[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    cat_sizes.append(len(levels) + 1)
Xc_tr = np.stack([codes_tr[c] for c in CAT_COLS], axis=1)
Xc_te = np.stack([codes_te[c] for c in CAT_COLS], axis=1)

mask_tr = np.isnan(train[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)
mask_te = np.isnan(test[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)

ratio block is a pure function of the features, not of y: True
row 106 fed 33 numeric + 9 mask columns
these arms feed 46 numeric + 9 mask


## The architecture

Written from the paper's description. The pieces that differ from row 127 are the periodic
embedding of every numeric feature, the learnable front scale, and the parameter groups that give
scale and bias tensors their own learning rate and weight decay.

In [6]:
import math


class PeriodicEmbedding(nn.Module):
    """Each numeric feature x_j becomes sin/cos of learnable frequencies, then its own
    linear map. This is the component row 127 has no equivalent of: it lets the network
    represent a non-monotone response to a single feature without spending depth on it."""

    def __init__(self, n_num, hidden=PB_HIDDEN, out=PB_OUT, freq_scale=PB_FREQ_SCALE):
        super().__init__()
        self.freq = nn.Parameter(torch.randn(n_num, hidden) * freq_scale)
        self.bias = nn.Parameter(torch.zeros(n_num, hidden))
        self.lin = nn.Parameter(torch.randn(n_num, 2 * hidden, out) / math.sqrt(2 * hidden))
        self.act = nn.PReLU(num_parameters=1)
        self.out_dim = n_num * out

    def forward(self, x):
        z = x.unsqueeze(-1) * self.freq + self.bias          # (B, n_num, hidden)
        e = torch.cat([torch.sin(z), torch.cos(z)], dim=-1)  # (B, n_num, 2*hidden)
        o = torch.einsum("bnh,nho->bno", e, self.lin)        # (B, n_num, out)
        return self.act(o).flatten(1)


class FrontScale(nn.Module):
    """A learnable diagonal scale on the raw numeric block. Cheap, and it lets the network
    down-weight a useless column without having to cancel it inside the first linear."""

    def __init__(self, n):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n))

    def forward(self, x):
        return x * self.scale


class RealMLP(nn.Module):
    def __init__(self, n_num, cat_sizes, hidden=HIDDEN, p=DROPOUT):
        super().__init__()
        self.front = FrontScale(n_num)
        self.periodic = PeriodicEmbedding(n_num)
        self.embs = nn.ModuleList([nn.Embedding(s, EMB_DIM) for s in cat_sizes])
        dim = n_num + self.periodic.out_dim + EMB_DIM * len(cat_sizes)
        layers = []
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(p)]
            dim = h
        layers.append(nn.Linear(dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, xn, xc):
        parts = [self.front(xn), self.periodic(xn)]
        parts += [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.net(torch.cat(parts, dim=1)).squeeze(1)


def param_groups(model):
    """Scale and bias tensors get their own learning rate and decay, as the paper specifies."""
    scale, bias, rest = [], [], []
    for n, prm in model.named_parameters():
        if "scale" in n or isinstance(prm, nn.Parameter) and prm.ndim == 1 and "bias" not in n:
            scale.append(prm)
        elif n.endswith("bias"):
            bias.append(prm)
        else:
            rest.append(prm)
    return [
        {"params": rest, "lr": LR, "weight_decay": WD},
        {"params": scale, "lr": LR * SCALE_LR_MULT, "weight_decay": WD * SCALE_WD_MULT},
        {"params": bias, "lr": LR * BIAS_LR_MULT, "weight_decay": WD * BIAS_WD_MULT},
    ]


class EMA:
    """Evaluate a moving average of the weights rather than the last iterate."""

    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items() if v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)

    def copy_to(self, model):
        sd = model.state_dict()
        for k, v in self.shadow.items():
            sd[k].copy_(v)


def flat_cos(step, total, flat=FLAT_RATIO):
    """Flat for the first `flat` fraction of training, then cosine to zero."""
    f = int(total * flat)
    if step < f:
        return 1.0
    return 0.5 * (1.0 + math.cos(math.pi * (step - f) / max(1, total - f)))


print("RealMLP defined. Components absent from row 127: periodic embeddings, front scale,")
print("parameter-group multipliers, EMA, flat-cosine schedule, label smoothing.")

RealMLP defined. Components absent from row 127: periodic embeddings, front scale,
parameter-group multipliers, EMA, flat-cosine schedule, label smoothing.


## The preprocessing the architecture specifies

Median-centre, IQR-scale, smooth-clip. This replaces row 127's quantile transform. All three are
fit on training rows only, inside the fold. `smooth_clip` is a soft bound that keeps extreme
values finite without the hard cut a clip would apply, which matters because the periodic
embedding is sensitive to the scale of its input.

In [7]:
def fit_prep(a):
    med = np.nanmedian(a, axis=0)
    med = np.where(np.isnan(med), 0.0, med)
    q1, q3 = np.nanpercentile(a, [25, 75], axis=0)
    iqr = np.where(np.isnan(q3 - q1) | ((q3 - q1) < 1e-9), 1.0, q3 - q1)
    return med, iqr


def apply_prep(a, med, iqr, c=4.0):
    x = np.where(np.isnan(a), med, a)
    x = (x - med) / iqr
    return (x / np.sqrt(1.0 + (x / c) ** 2)).astype(np.float32)   # smooth clip


NUM_WORKERS = 0 if os.name == "nt" else 2


def make_loader(xn, xc, yy, bs, shuffle, drop_last=False):
    ds = torch.utils.data.TensorDataset(
        torch.from_numpy(xn), torch.from_numpy(xc),
        torch.from_numpy(yy) if yy is not None else torch.zeros(len(xn)))
    return torch.utils.data.DataLoader(
        ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=(DEV.type == "cuda"), drop_last=drop_last)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []
    for xn, xc, _ in loader:
        out.append(torch.sigmoid(model(xn.to(DEV), xc.to(DEV))).float().cpu().numpy())
    return np.concatenate(out)


LOG = OUT / "57_realmlp_steps.log"


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== probe start, SMOKE={SMOKE}, device={DEV}, fold={PROBE_FOLD}, "
     f"n_ens={PROBE_ENS} ===")

=== probe start, SMOKE=True, device=cpu, fold=0, n_ens=2 ===


In [8]:
# Fold-0 probe. One fold, four arms, everything else held at row 138's configuration.
# No out-of-fold vector is written and no ledger row comes out of this notebook.
f = PROBE_FOLD
tr_i = np.where(folds != f)[0]
va_i = np.where(folds == f)[0]

Etr, Eva, Ete = build(X, y, tr_i, va_i, X_test)
num_tr = np.hstack([Etr[NUM_COLS + ENC_COLS].to_numpy(np.float32), RB_TR[tr_i]])
num_va = np.hstack([Eva[NUM_COLS + ENC_COLS].to_numpy(np.float32), RB_TR[va_i]])
med, iqr = fit_prep(num_tr)
Xn_tr = np.hstack([apply_prep(num_tr, med, iqr), mask_tr[tr_i]])
Xn_va = np.hstack([apply_prep(num_va, med, iqr), mask_tr[va_i]])
note(f"numeric block {Xn_tr.shape[1]} columns, {len(tr_i):,} train / {len(va_i):,} valid")

t_start = time.time()
probe = {}
for arm, bs, eps_n in ARM_GRID:
    tr_loader = make_loader(Xn_tr, Xc_tr[tr_i], y[tr_i], bs, True, drop_last=True)
    va_loader = make_loader(Xn_va, Xc_tr[va_i], None, bs * 8, False)
    steps_total = eps_n * len(tr_loader)

    va_ens = np.zeros(len(va_i))
    singles = []
    for e in range(PROBE_ENS):
        # Same seed stream as row 138 so arm A is that configuration and not a re-roll.
        seed_all(SEED + 100 * f + e)
        model = RealMLP(Xn_tr.shape[1], cat_sizes).to(DEV)
        opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.98))
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: flat_cos(s, steps_total))
        ema = EMA(model)
        for ep in range(eps_n):
            model.train()
            eps = LS_EPS * 0.5 * (1 + math.cos(math.pi * ep / max(1, eps_n - 1)))
            for xn, xc, yy in tr_loader:
                yy = yy.to(DEV)
                yy = yy * (1 - eps) + 0.5 * eps
                opt.zero_grad(set_to_none=True)
                loss = nn.functional.binary_cross_entropy_with_logits(
                    model(xn.to(DEV), xc.to(DEV)), yy)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()
                sched.step()
                ema.update(model)
        ema.copy_to(model)
        pv = predict(model, va_loader)
        singles.append(float(roc_auc_score(y[va_i], pv)))
        va_ens += pv / PROBE_ENS
        del model, ema
        if DEV.type == "cuda":
            torch.cuda.empty_cache()

    probe[arm] = {"auc": float(roc_auc_score(y[va_i], va_ens)), "steps": steps_total,
                  "batch": bs, "epochs": eps_n, "singles": singles}
    note(f"{arm}: ens AUC {probe[arm]['auc']:.6f}  singles "
         f"{' '.join(f'{s:.6f}' for s in singles)}  steps {steps_total:,}  "
         f"elapsed {(time.time()-t_start)/60:.1f} min")


numeric block 55 columns, 16,000 train / 4,000 valid


A_b512_e12: ens AUC 0.921997  singles 0.872498 0.921449  steps 31  elapsed 0.1 min


B_b256_e12: ens AUC 0.935939  singles 0.931236 0.926513  steps 62  elapsed 0.2 min


C_b512_e24: ens AUC 0.935520  singles 0.896957 0.929475  steps 62  elapsed 0.3 min


D_b256_e24: ens AUC 0.939518  singles 0.937261 0.933163  steps 124  elapsed 0.5 min


In [9]:
base = probe["A_b512_e12"]["auc"]
print(f"fold {PROBE_FOLD}, {PROBE_ENS} internal members per arm, paired on identical rows\n")
print(f"{'arm':12} {'batch':>6} {'epochs':>7} {'steps':>9} {'ens AUC':>10} {'vs A':>10}")
for a, _, _ in ARM_GRID:
    p = probe[a]
    print(f"{a:12} {p['batch']:6d} {p['epochs']:7d} {p['steps']:9,d} "
          f"{p['auc']:10.6f} {p['auc'] - base:+10.6f}")

print(f"\narm A is row 138's configuration. Row 138 scored {ROW138_FOLD0:.6f} on this fold at")
print(f"ten internal members; this probe runs {PROBE_ENS}, so arm A should sit a little below it.")
if not SMOKE:
    d = base - ROW138_FOLD0
    print(f"arm A reads {base:.6f}, delta {d:+.6f}. "
          + ("consistent with the smaller ensemble" if -0.003 < d < 0.001
             else "OUTSIDE the expected band, treat this probe as unexplained"))

print("\nSEPARATING BATCH FROM STEP COUNT:")
bs_effect = (probe["B_b256_e12"]["auc"] - probe["A_b512_e12"]["auc"]
             + probe["D_b256_e24"]["auc"] - probe["C_b512_e24"]["auc"]) / 2
ep_effect = (probe["C_b512_e24"]["auc"] - probe["A_b512_e12"]["auc"]
             + probe["D_b256_e24"]["auc"] - probe["B_b256_e12"]["auc"]) / 2
print(f"  halving the batch, averaged over both epoch settings: {bs_effect:+.6f}")
print(f"  doubling the epochs, averaged over both batch settings: {ep_effect:+.6f}")
print("  If these are close, the lever is STEP COUNT and either route reaches it.")
print("  If halving the batch dominates, it is gradient noise and epochs will not substitute.")

best = max(probe, key=lambda a: probe[a]["auc"])
gain = probe[best]["auc"] - base
print(f"\nbest arm: {best} at {probe[best]['auc']:.6f}, {gain:+.6f} over row 138's setting")
print(f"public RealMLP is about 0.9688 five-fold; row 138 is {ROW138_CV:.6f}, a gap of ~0.0009.")
print("A probe difference under about 0.0003 should not be trusted (rows 9 to 12).")
if gain < 0.0003:
    print("\nVERDICT: under the trust threshold. The batch/step axis does not explain the gap.")
else:
    print(f"\nVERDICT: worth a five-fold run of {best} as a one-variable row against 138.")
print("\nNo ledger row, no OOF vector written. This notebook prices an axis, it does not log one.")


fold 0, 2 internal members per arm, paired on identical rows

arm           batch  epochs     steps    ens AUC       vs A
A_b512_e12      512       1        31   0.921997  +0.000000
B_b256_e12      256       1        62   0.935939  +0.013942
C_b512_e24      512       2        62   0.935520  +0.013523
D_b256_e24      256       2       124   0.939518  +0.017521

arm A is row 138's configuration. Row 138 scored 0.967220 on this fold at
ten internal members; this probe runs 2, so arm A should sit a little below it.

SEPARATING BATCH FROM STEP COUNT:
  halving the batch, averaged over both epoch settings: +0.008970
  doubling the epochs, averaged over both batch settings: +0.008551
  If these are close, the lever is STEP COUNT and either route reaches it.
  If halving the batch dominates, it is gradient noise and epochs will not substitute.

best arm: D_b256_e24 at 0.939518, +0.017521 over row 138's setting
public RealMLP is about 0.9688 five-fold; row 138 is 0.967892, a gap of ~0.0009.
A p

In [10]:
# Deliberately empty. A probe writes no vector and no submission.
print("probe complete, nothing written")


probe complete, nothing written
